<a href="https://colab.research.google.com/github/Herculano-Chitungo/nexa-tech-etl-dashboard/blob/main/Notebook_ETL_NexaTech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Grupo 8 - Gestão de Funcionários

import pandas as pd
import numpy as np
import io
import zipfile
import os
from datetime import datetime
from google.colab import files

print("Faça o upload dos 6 ficheiros CSV originais exportados do DBeaver:")
uploaded = files.upload()

def carregar_csv(termos):
    if isinstance(termos, str):
        termos = [termos]
    for termo in termos:
        match = [f for f in uploaded.keys() if termo.lower() in f.lower()]
        if match:
            return pd.read_csv(match[0])
    return None

# Carregar as tabelas originais
df_func = carregar_csv('Funcionarios')
df_dept = carregar_csv('Departamentos')
df_filiais = carregar_csv(['Filiais', 'Localizacao'])
df_cargos = carregar_csv(['Cargos', 'Senioridade'])
df_custos = carregar_csv(['Custos', 'Despesas'])
df_aval = carregar_csv(['Avaliacao', 'ROI'])

# 1. Limpeza de espaços em branco em strings e padronização da Cor da Pele
for df in [df_func, df_dept, df_filiais, df_cargos, df_aval]:
    if df is not None:
        for col in df.select_dtypes(include=['object']).columns:
            df[col] = df[col].astype(str).str.strip()

if 'Cor_Pele' in df_func.columns:
    df_func['Cor_Pele'] = df_func['Cor_Pele'].replace({'Negro': 'Preto', 'negro': 'preto'})

# 2. Adicionar filial no Rio de Janeiro (RJ) e novo departamento de Operações RJ
nova_filial = pd.DataFrame({
    'Filial_ID': [4],
    'Nome_Cidade': ['Rio de Janeiro'],
    'Estado': ['RJ'],
    'Tipo_Escritorio': ['Escritório Comercial RJ']
})
df_filiais = pd.concat([df_filiais, nova_filial], ignore_index=True)

novo_depto = pd.DataFrame({
    'Codigo': [88],
    'Nome': ['Operações RJ'],
    'Orcamento_Total': [450000.0],
    'Filial_ID': [4]
})
df_dept = pd.concat([df_dept, novo_depto], ignore_index=True)

# Alocar funcionários para o Rio de Janeiro
np.random.seed(42)
funcionarios_rj_idx = np.random.choice(df_func.index, size=15, replace=False)
df_func.loc[funcionarios_rj_idx, 'Departamento_ID'] = 88
df_func.loc[funcionarios_rj_idx, 'Morada'] = 'Rio de Janeiro'

# 3. Personalizar salários por funcionário
base_salarios = {1: 3800.0, 2: 6800.0, 3: 11500.0, 4: 16000.0}
df_func['Salario_Base'] = df_func['Cargo_ID'].map(base_salarios)
variacao = np.random.uniform(0.92, 1.15, size=len(df_func))
df_func['Salario_Base'] = (df_func['Salario_Base'] * variacao).round(-2)

# 4. Avaliações completas para TODOS os funcionários (105 registos)
anos_ref = [2024, 2025, 2026]
nova_aval = []
for idx, row in df_func.iterrows():
    score = round(np.random.uniform(7.0, 9.9), 1)
    base_roi = row['Cargo_ID'] * 25000
    roi = round(base_roi * np.random.uniform(0.8, 1.4), -2)
    ano = np.random.choice(anos_ref, p=[0.2, 0.3, 0.5])
    nova_aval.append({
        'Avaliacao_ID': idx + 1,
        'CPF_Funcionario': row['CPF'],
        'Score_Produtividade': score,
        'Retorno_Estimado_ROI': roi,
        'Ano_Referencia': ano
    })
df_aval = pd.DataFrame(nova_aval)

# 5. Expandir custos para 24 meses (Janeiro 2025 a Dezembro 2026)
meses = pd.date_range(start='2025-01-01', end='2026-12-01', freq='MS').strftime('%Y-%m').tolist()
novos_custos = []
despesa_id_counter = 1
deptos_lista = df_dept['Codigo'].tolist()

for mes in meses:
    for d_id in deptos_lista:
        base_mult = 1.0 if d_id != 88 else 1.1
        if d_id == 14: base_mult = 1.5
        elif d_id == 59: base_mult = 1.7
        elif d_id == 77: base_mult = 1.3

        fator_tempo = 1.0 if mes.startswith('2025') else 1.05

        for cat, val_base in [('Custos Fixos', 70000.0), ('Benefícios', 50000.0), ('Equipamentos', 30000.0), ('Formação', 15000.0)]:
            novos_custos.append({
                'Despesa_ID': despesa_id_counter,
                'Departamento_ID': d_id,
                'Categoria_Despesa': cat,
                'Valor': round(val_base * base_mult * fator_tempo * np.random.uniform(0.92, 1.08), -2),
                'Mes_Referencia': f"{mes}-01"
            })
            despesa_id_counter += 1

df_custos = pd.DataFrame(novos_custos)

# 6. Criar a Tabela Dimensão de Departamentos com IDs ÚNICOS
if df_dept is not None:
    if 'Codigo' in df_dept.columns:
        df_dept.rename(columns={'Codigo': 'Departamento_ID'}, inplace=True)
    df_dept['Orcamento_Total'] = pd.to_numeric(df_dept['Orcamento_Total'].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')
    dim_departamentos = df_dept[['Departamento_ID', 'Nome', 'Orcamento_Total', 'Filial_ID']].drop_duplicates(subset=['Departamento_ID'])
else:
    dim_departamentos = None

# 7. Preparar a tabela Master de Funcionários
if df_func is not None and dim_departamentos is not None and df_cargos is not None and df_filiais is not None:
    for col_rem in ['Orcamento_Total', 'Nome_Departamento']:
        if col_rem in df_func.columns:
            df_func = df_func.drop(columns=[col_rem])

    df_master = pd.merge(df_func, dim_departamentos[['Departamento_ID', 'Nome', 'Filial_ID']], on='Departamento_ID', how='inner')
    df_master = pd.merge(df_master, df_cargos, on='Cargo_ID', how='inner')
    df_master = pd.merge(df_master, df_filiais, on='Filial_ID', how='inner')

    if df_aval is not None:
        if 'CPF_Funcionario' in df_aval.columns:
            df_master = pd.merge(df_master, df_aval, left_on='CPF', right_on='CPF_Funcionario', how='left')
            if 'CPF_Funcionario' in df_master.columns:
                df_master.drop(columns=['CPF_Funcionario'], inplace=True)

    df_master['Data_Admissao'] = pd.to_datetime(df_master['Data_Admissao'], errors='coerce')
    data_atual = pd.to_datetime('today')
    df_master['Tempo_Empresa_Anos'] = ((data_atual - df_master['Data_Admissao']).dt.days / 365.25).round(1)
    df_master['Ano_Admissao'] = df_master['Data_Admissao'].dt.year

    bins_idade = [18, 24, 34, 45, 100]
    labels_idade = ['18-24 anos', '25-34 anos', '35-44 anos', '45+ anos']
    df_master['Faixa_Etaria'] = pd.cut(df_master['Idade'], bins=bins_idade, labels=labels_idade, right=True)

    contagem_dept = df_master['Departamento_ID'].value_counts().to_dict()
    df_master['Qtd_Funcionarios_Depto'] = df_master['Departamento_ID'].map(contagem_dept)

    mapa_orcamento = dim_departamentos.set_index('Departamento_ID')['Orcamento_Total'].to_dict()
    df_master['Orcamento_Dept_Ref'] = df_master['Departamento_ID'].map(mapa_orcamento)
    df_master['Media_Verba_Por_Funcionario'] = (df_master['Orcamento_Dept_Ref'] / df_master['Qtd_Funcionarios_Depto']).round(2)
    df_master.drop(columns=['Orcamento_Dept_Ref'], inplace=True)

    if df_custos is not None:
        df_custos['Valor'] = pd.to_numeric(df_custos['Valor'].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')

    # 8. Exportar os ficheiros finais
    dim_departamentos.to_csv('dimensao_departamentos.csv', index=False, encoding='utf-8-sig')
    df_master.to_csv('dados_funcionarios_master.csv', index=False, encoding='utf-8-sig')
    if df_custos is not None:
        df_custos.to_csv('dados_custos_despesas.csv', index=False, encoding='utf-8-sig')

    # Criar arquivo ZIP automático para download único
    zip_filename = 'Tabelas_Trabalhadas_PowerBI_NexaTech.zip'
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write('dimensao_departamentos.csv')
        zipf.write('dados_funcionarios_master.csv')
        zipf.write('dados_custos_despesas.csv')

    print("\n--- SUCESSO! Dados tratados e ZIP gerados com sucesso. ---")
    files.download(zip_filename)
else:
    print("\n[ERRO] Não foi possível carregar as tabelas principais.")

Faça o upload dos 6 ficheiros CSV originais exportados do DBeaver:


Saving Avaliacao_Desempenho_ROI_202610082216.csv to Avaliacao_Desempenho_ROI_202610082216.csv
Saving Cargos_Senioridade_202610082217.csv to Cargos_Senioridade_202610082217.csv
Saving Departamentos_202610082217.csv to Departamentos_202610082217.csv
Saving Filiais_Localizacao_202610082217.csv to Filiais_Localizacao_202610082217.csv
Saving Funcionarios_202610082217.csv to Funcionarios_202610082217.csv
Saving Historico_Custos_Despesas_202610082217.csv to Historico_Custos_Despesas_202610082217.csv

--- SUCESSO! Dados tratados e ZIP gerados com sucesso. ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>